# Morogoro Price Forecasting Final Model

This notebook trains one final regression model for all Morogoro rice and beans series using:

- `morogoro_rice_beans_updated.csv`

Selected model:
- `RandomForestRegressor`
- trained with stronger sample weights on higher-price periods so the predicted curve stays closer to peak prices

This notebook:
- trains one clean final model per series using the same algorithm
- focuses on both overall accuracy and high-price fit
- saves the final model bundle and output files
- plots 3 clean regression graphs for each series key

Saved outputs after running:
- `morogoro_price_forecaster_final.joblib`
- `morogoro_price_model_metrics_final.csv`
- `morogoro_price_validation_predictions_final.csv`

In [ ]:
import os, sys, django
try:
    sys.path.insert(0, str(BASE_DIR.parent if BASE_DIR.name == 'ai' else BASE_DIR))
    os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings')
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', 'true')
    django.setup()
    from apps.markets.models import MarketCommodityPrice
    qs = MarketCommodityPrice.objects.filter(deleted_at__isnull=True).values(
        'price_date', 'commodity__name', 'unit__symbol', 'price_type', 'currency', 'price'
    )
    if qs.exists():
        df = pd.DataFrame.from_records(qs)
        df.rename(columns={'price_date': 'date', 'commodity__name': 'commodity', 'unit__symbol': 'unit', 'price_type': 'pricetype'}, inplace=True)
        df['date'] = pd.to_datetime(df['date'])
        df['pricetype'] = df['pricetype'].astype(str).str.title()
        print(f'Loaded {len(df)} records directly from Django database.')
    else:
        df = pd.read_csv(DATASET_PATH, parse_dates=['date'])
except Exception as err:
    df = pd.read_csv(DATASET_PATH, parse_dates=['date'])
    print(f'Loaded records from CSV fallback: {err}')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['date', 'price']).copy()
df['commodity'] = df['commodity'].str.strip()
df['unit'] = df['unit'].str.strip()
df['pricetype'] = df['pricetype'].str.strip()


In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=["date"])
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df = df.dropna(subset=["date", "price"]).copy()
df["commodity"] = df["commodity"].str.strip()
df["unit"] = df["unit"].str.strip()
df["pricetype"] = df["pricetype"].str.strip()

series_df = (
    df.groupby(["date", "commodity", "unit", "pricetype"], as_index=False)
    .agg(
        price=("price", "mean"),
        region=("region", "first"),
        category=("category", "first"),
        currency=("currency", "first"),
    )
    .sort_values(["commodity", "unit", "pricetype", "date"])
    .reset_index(drop=True)
)

series_df["series_key"] = (
    series_df["commodity"] + "|" + series_df["unit"] + "|" + series_df["pricetype"]
)

series_df.groupby("series_key").agg(rows=("price", "size"), start_date=("date", "min"), end_date=("date", "max"))

In [ ]:
def add_calendar_features(frame: pd.DataFrame) -> pd.DataFrame:
    enriched = frame.copy()
    enriched["year"] = enriched["date"].dt.year
    enriched["month"] = enriched["date"].dt.month
    enriched["day"] = enriched["date"].dt.day
    enriched["dayofweek"] = enriched["date"].dt.dayofweek
    enriched["dayofyear"] = enriched["date"].dt.dayofyear
    enriched["weekofyear"] = enriched["date"].dt.isocalendar().week.astype(int)
    enriched["quarter"] = enriched["date"].dt.quarter
    enriched["month_sin"] = np.sin(2 * np.pi * enriched["month"] / 12)
    enriched["month_cos"] = np.cos(2 * np.pi * enriched["month"] / 12)
    enriched["dayofyear_sin"] = np.sin(2 * np.pi * enriched["dayofyear"] / 365.25)
    enriched["dayofyear_cos"] = np.cos(2 * np.pi * enriched["dayofyear"] / 365.25)
    return enriched

def build_training_frame(series_frame: pd.DataFrame) -> pd.DataFrame:
    train = series_frame[["date", "price"]].sort_values("date").copy()
    train = add_calendar_features(train)
    train["lag_1"] = train["price"].shift(1)
    train["lag_3"] = train["price"].shift(3)
    train["lag_7"] = train["price"].shift(7)
    train["lag_14"] = train["price"].shift(14)
    train["lag_30"] = train["price"].shift(30)
    train["rolling_7"] = train["price"].shift(1).rolling(7).mean()
    train["rolling_14"] = train["price"].shift(1).rolling(14).mean()
    train["rolling_30"] = train["price"].shift(1).rolling(30).mean()
    train["rolling_max_30"] = train["price"].shift(1).rolling(30).max()
    train["rolling_min_30"] = train["price"].shift(1).rolling(30).min()
    train["rolling_std_30"] = train["price"].shift(1).rolling(30).std()
    train["expanding_mean"] = train["price"].shift(1).expanding().mean()
    train["pct_change_1"] = train["price"].pct_change().shift(1)
    train["pct_change_7"] = train["price"].pct_change(7).shift(1)
    return train.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

FEATURE_COLUMNS = [
    "year",
    "month",
    "day",
    "dayofweek",
    "dayofyear",
    "weekofyear",
    "quarter",
    "month_sin",
    "month_cos",
    "dayofyear_sin",
    "dayofyear_cos",
    "lag_1",
    "lag_3",
    "lag_7",
    "lag_14",
    "lag_30",
    "rolling_7",
    "rolling_14",
    "rolling_30",
    "rolling_max_30",
    "rolling_min_30",
    "rolling_std_30",
    "expanding_mean",
    "pct_change_1",
    "pct_change_7",
]

def create_model() -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    )

def evaluate_predictions(actual, predicted):
    mae = float(mean_absolute_error(actual, predicted))
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    r2 = float(r2_score(actual, predicted))
    mape = float(mean_absolute_percentage_error(actual, predicted) * 100)
    accuracy_from_mape = float(max(0.0, 100.0 - mape))
    return mae, rmse, r2, mape, accuracy_from_mape

In [ ]:
model_bundle = {
    "dataset_filename": DATASET_FILENAME,
    "feature_columns": FEATURE_COLUMNS,
    "model_name": "random_forest_weighted",
    "models": {},
    "series_metadata": {},
}

metrics_rows = []
validation_rows = []

for series_key, group in series_df.groupby("series_key"):
    group = group.sort_values("date").reset_index(drop=True)
    training_frame = build_training_frame(group)
    if len(training_frame) < 120:
        print(f"Skipping {series_key}: not enough rows after feature engineering.")
        continue

    split_index = max(int(len(training_frame) * 0.8), 1)
    train_split = training_frame.iloc[:split_index].copy()
    test_split = training_frame.iloc[split_index:].copy()

    X_train = train_split[FEATURE_COLUMNS]
    y_train = train_split["price"]
    X_test = test_split[FEATURE_COLUMNS]
    y_test = test_split["price"]

    train_q85 = y_train.quantile(0.85)
    train_q95 = y_train.quantile(0.95)
    sample_weights = np.where(y_train >= train_q95, 5.0, np.where(y_train >= train_q85, 3.0, 1.0))

    test_high_threshold = y_test.quantile(0.9)
    high_mask_test = y_test >= test_high_threshold

    model = create_model()
    model.fit(X_train, y_train, sample_weight=sample_weights)
    predictions = model.predict(X_test)

    mae, rmse, r2, mape, accuracy_from_mape = evaluate_predictions(y_test, predictions)
    high_mae, high_rmse, high_r2, high_mape, high_accuracy_from_mape = evaluate_predictions(y_test[high_mask_test], predictions[high_mask_test])
    high_bias = float((predictions[high_mask_test] - y_test[high_mask_test]).mean())

    final_model = create_model()
    full_q85 = training_frame["price"].quantile(0.85)
    full_q95 = training_frame["price"].quantile(0.95)
    full_weights = np.where(training_frame["price"] >= full_q95, 5.0, np.where(training_frame["price"] >= full_q85, 3.0, 1.0))
    final_model.fit(training_frame[FEATURE_COLUMNS], training_frame["price"], sample_weight=full_weights)

    model_bundle["models"][series_key] = final_model
    model_bundle["series_metadata"][series_key] = {
        "commodity": group.iloc[0]["commodity"],
        "unit": group.iloc[0]["unit"],
        "pricetype": group.iloc[0]["pricetype"],
        "currency": group.iloc[0]["currency"],
        "start_date": str(group["date"].min().date()),
        "end_date": str(group["date"].max().date()),
        "sample_weight_rule": "price >= q95 => 5.0, price >= q85 => 3.0, else 1.0",
    }

    metrics_rows.append(
        {
            "series_key": series_key,
            "commodity": group.iloc[0]["commodity"],
            "unit": group.iloc[0]["unit"],
            "pricetype": group.iloc[0]["pricetype"],
            "rows": len(group),
            "mae": round(mae, 4),
            "rmse": round(rmse, 4),
            "r2": round(r2, 4),
            "mape_percent": round(mape, 4),
            "accuracy_from_mape_percent": round(accuracy_from_mape, 4),
            "high_price_mae": round(high_mae, 4),
            "high_price_rmse": round(high_rmse, 4),
            "high_price_mape_percent": round(high_mape, 4),
            "high_price_accuracy_from_mape_percent": round(high_accuracy_from_mape, 4),
            "high_price_bias": round(high_bias, 4),
        }
    )

    validation_rows.append(
        pd.DataFrame(
            {
                "series_key": series_key,
                "date": test_split["date"].to_numpy(),
                "actual_price": y_test.to_numpy(),
                "predicted_price": predictions,
                "residual": y_test.to_numpy() - predictions,
                "is_high_price": high_mask_test.to_numpy(),
            }
        )
    )

metrics_df = pd.DataFrame(metrics_rows).sort_values(["commodity", "unit", "pricetype"]).reset_index(drop=True)
validation_predictions_df = pd.concat(validation_rows, ignore_index=True).sort_values(["series_key", "date"]).reset_index(drop=True)

joblib.dump(model_bundle, MODEL_OUTPUT_PATH)
metrics_df.to_csv(METRICS_OUTPUT_PATH, index=False)
validation_predictions_df.to_csv(PREDICTIONS_OUTPUT_PATH, index=False)

print(f"Saved model artifact: {MODEL_OUTPUT_PATH}")
print(f"Saved metrics: {METRICS_OUTPUT_PATH}")
print(f"Saved validation predictions: {PREDICTIONS_OUTPUT_PATH}")
metrics_df

In [ ]:
for series_key in metrics_df["series_key"]:
    plot_df = validation_predictions_df[validation_predictions_df["series_key"] == series_key].copy()
    high_df = plot_df[plot_df["is_high_price"]]

    plt.figure(figsize=(7, 6))
    plt.scatter(plot_df["actual_price"], plot_df["predicted_price"], alpha=0.35)
    line_min = min(plot_df["actual_price"].min(), plot_df["predicted_price"].min())
    line_max = max(plot_df["actual_price"].max(), plot_df["predicted_price"].max())
    plt.plot([line_min, line_max], [line_min, line_max], color="red", linestyle="--")
    plt.title(f"Actual vs Predicted: {series_key}")
    plt.xlabel("Actual Price")
    plt.ylabel("Predicted Price")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.scatter(plot_df["predicted_price"], plot_df["residual"], alpha=0.35)
    plt.axhline(0, color="red", linestyle="--")
    plt.title(f"Residuals vs Predicted: {series_key}")
    plt.xlabel("Predicted Price")
    plt.ylabel("Residual (Actual - Predicted)")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(plot_df["date"], plot_df["actual_price"], label="Actual Price")
    plt.plot(plot_df["date"], plot_df["predicted_price"], label="Predicted Price", linestyle="--")
    plt.scatter(high_df["date"], high_df["actual_price"], color="orange", s=14, label="High Price Periods")
    plt.title(f"Actual vs Predicted Over Time: {series_key}")
    plt.xlabel("Date")
    plt.ylabel("Price")
    plt.legend()
    plt.tight_layout()
    plt.show()